In [22]:
import argparse
import pandas as pd

In [23]:
PRESENTATION_START = {
    "2013B": "2013-02-01",
    "2013J": "2013-10-01",
    "2014B": "2014-02-01",
    "2014J": "2014-10-01",
    }

In [24]:
# Cohort mana yang dianggap "churn resmi" vs "masih terdaftar tapi pasif"
# data_unregistration kosong (NaN) artinya siswa tidak pernah keluar secara resmi.

def load_data(data_dir: str) -> dict[str, pd.DataFrame]:
  files = {
      "course": "courses.csv",
      "student_info": "studentInfo.csv",
      "registration": "studentRegistration.csv",
      "vle": "studentVle.csv",
  }
  data = {}
  for key, fname in files.items():
    path = f"{data_dir.rstrip('/')}/{fname}"
    data[key] = pd.read_csv(path)
    print(f"Loaded {fname}: {data[key].shape[0]:,} rows")
  return data

In [25]:
def build_cohort_base(student_info: pd.DataFrame, registration: pd.DataFrame) -> pd.DataFrame:
  base = student_info.merge(
      registration,
      on=["code_module", "code_presentation", "id_student"],
      how="left"
  )

  base["cohort_start"] = base["code_presentation"].map(PRESENTATION_START)
  base["cohort_start"] = pd.to_datetime(base["cohort_start"])

  withdrew_before_start = base["date_unregistration"] < 0
  n_dropped = withdrew_before_start.sum()
  if n_dropped:
    print(f"Excluding {n_dropped:,} students who unregistered before day 0")
  base = base[~withdrew_before_start].copy()
  return base

In [26]:
def compute_weekly_activity(vle: pd.DataFrame, max_week: int = 30) -> pd.DataFrame:
  vle = vle.copy()
  vle["week_number"] = vle["date"] // 7
  vle = vle[(vle["week_number"] >= 0) & (vle["week_number"] <= max_week)]

  weekly = (
      vle.groupby(["code_module", "code_presentation", "id_student", "week_number"])["sum_click"].sum().reset_index())

  weekly["is_active"] = weekly["sum_click"] > 0
  return weekly[weekly["is_active"]][["code_module", "code_presentation", "id_student", "week_number"]]

In [27]:
def build_cohort_retention_table(base: pd.DataFrame, weekly_active: pd.DataFrame, max_week: int = 30) -> pd.DataFrame:
  cohort_sizes = (
      base.groupby(["code_module", "code_presentation"])["id_student"]
      .nunique()
      .reset_index(name="cohort_size")
  )

  weeks = pd.DataFrame({"week_number": range(0, max_week + 1)})
  cohort_weeks = cohort_sizes.merge(weeks, how="cross")

  active_counts = (
      weekly_active.groupby(["code_module", "code_presentation", "week_number"])["id_student"]
      .nunique()
      .reset_index(name="active_students")
  )

  result = cohort_weeks.merge(
      active_counts,
      on=["code_module", "code_presentation", "week_number"],
      how="left"
  )

  result["active_students"] = result["active_students"].fillna(0).astype(int)
  result["retention_pct"] = (result["active_students"] / result["cohort_size"] * 100).round(1)

  result["cohort_label"] = result["code_module"] + " · " + result["code_presentation"]

  return result.sort_values(["code_module", "code_presentation", "week_number"])

In [32]:
def main():
  parser = argparse.ArgumentParser(description="Build OULAD cohort retention table")
  parser.add_argument("--data_dir", default=".", help="Folder berisi CSV OULAD")
  parser.add_argument("--out", default="cohort_retention.csv", help="Path file output")
  parser.add_argument("--max_week", default=30, type=int, help="Batas minggu ke-N yang dianalisis")

  args = parser.parse_args([])

  data = load_data(args.data_dir)
  base = build_cohort_base(data["student_info"], data["registration"])
  weekly_active = compute_weekly_activity(data["vle"], max_week = args.max_week)
  retention = build_cohort_retention_table(base, weekly_active, max_week = args.max_week)
  retention.to_csv(args.out, index=False)
  print(f"\nSaved {len(retention):,} rows -> {args.out}")
  print(f"Cohorts: {retention['cohort_label'].nunique()}")
  print(retention.head(10).to_string(index=False))

In [33]:
if __name__ == "__main__":
  main()


Loaded courses.csv: 22 rows
Loaded studentInfo.csv: 32,593 rows
Loaded studentRegistration.csv: 32,593 rows
Loaded studentVle.csv: 10,655,280 rows
Excluding 2,678 students who unregistered before day 0

Saved 682 rows -> cohort_retention.csv
Cohorts: 22
code_module code_presentation  cohort_size  week_number  active_students  retention_pct cohort_label
        AAA             2013J          376            0              337           89.6  AAA · 2013J
        AAA             2013J          376            1              341           90.7  AAA · 2013J
        AAA             2013J          376            2              357           94.9  AAA · 2013J
        AAA             2013J          376            3              305           81.1  AAA · 2013J
        AAA             2013J          376            4              315           83.8  AAA · 2013J
        AAA             2013J          376            5              312           83.0  AAA · 2013J
        AAA             2013J          